In [2]:
# We use -U to ensure we have the latest versions of LangChain and Google GenAI
# packages to avoid compatibility errors.
!pip install python-dotenv rank-bm25 sentence-transformers langchain-google-genai langchain-community -U --quiet

print("Installation complete. Please restart your session if prompted by Colab.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
Installation complete. Please restart your session if prompted by Colab.


In [10]:
# Cell 1: Imports and Secure API Key Entry
import os
import getpass
import numpy as np

# Retrieval & NLP Libraries
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# LLM Framework (LangChain)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Securely prompt for the Google API Key
# This will create a text box for you to paste your key into.
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API Key here (it will be hidden): ")

# Verify the key is loaded
if os.environ["GOOGLE_API_KEY"].startswith("AIza"):
    print("Setup Complete. API Key loaded and libraries ready.")
else:
    print("❌ Warning: The API Key entered does not look valid. Please re-run this cell.")

Setup Complete. API Key loaded and libraries ready.


In [11]:
# A collection of 10 documents covering AI/ML topics to test our RAG system.
corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel.", # doc_0
    "BERT is a bidirectional encoder trained using masked language modelling.",      # doc_1
    "The BM25 algorithm ranks documents based on term frequency and inverse document frequency.", # doc_2
    "Gradient descent is an optimization technique used to minimize the loss function.", # doc_3
    "Neural networks learn by adjusting weights through backpropagation.",           # doc_4
    "Backpropagation calculates gradients to update weights in a neural network during training.", # doc_5 (Related to doc_4)
    "Stochastic Gradient Descent (SGD) is a variation of gradient descent that updates weights using a single sample.", # doc_6 (Related to doc_3)
    "The GELU activation function is often used in modern transformer architectures like GPT and BERT.", # doc_7 (Technical Jargon/Proper Nouns)
    "Fine-tuning adapts a pre-trained model to a specific downstream task using task-specific data.", # doc_8
    "Retrieval Augmented Generation combines a retriever with a language model to produce grounded answers." # doc_9
]

print(f"Corpus initialized with {len(corpus)} documents.")
print(f"Sample Document [7]: {corpus[7]}")

Corpus initialized with 10 documents.
Sample Document [7]: The GELU activation function is often used in modern transformer architectures like GPT and BERT.


In [13]:
# Cell 3: Hybrid Retriever Implementation (Part 2)
class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        """
        Initializes the Hybrid Retriever with a corpus and a smoothing constant k.
        """
        self.corpus = corpus
        self.k = k

        # 1. Initialize BM25 (Sparse/Keyword)
        # We tokenize by lowercasing and splitting on whitespace as per the hints.
        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        # 2. Initialize SBERT (Dense/Semantic)
        # We use the 'all-MiniLM-L6-v2' model for efficient vector embeddings.
        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.doc_embeddings = self.sbert.encode(corpus, convert_to_numpy=True)

        # Normalize embeddings to simplify cosine similarity to a dot product.
        self.doc_embeddings = self.doc_embeddings / np.linalg.norm(self.doc_embeddings, axis=1, keepdims=True)

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        """
        Performs Hybrid Retrieval using Reciprocal Rank Fusion (RRF).
        """
        # --- BM25 Scoring & Ranking ---
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked_idx = np.argsort(bm25_scores)[::-1]
        # Create a mapping of {doc_id: rank} where rank starts at 1
        bm25_ranks = {int(idx): rank + 1 for rank, idx in enumerate(bm25_ranked_idx)}

        # --- SBERT Scoring & Ranking ---
        q_emb = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_emb = q_emb / np.linalg.norm(q_emb) # Normalize query vector
        sbert_scores = self.doc_embeddings @ q_emb
        sbert_ranked_idx = np.argsort(sbert_scores)[::-1]
        # Create a mapping of {doc_id: rank} where rank starts at 1
        sbert_ranks = {int(idx): rank + 1 for rank, idx in enumerate(sbert_ranked_idx)}

        # --- Reciprocal Rank Fusion (RRF) ---
        final_results = []
        for i in range(len(self.corpus)):
            # RRF Formula: 1 / (k + rank_bm25) + 1 / (k + rank_sbert)
            rrf_score = (1.0 / (self.k + bm25_ranks[i])) + (1.0 / (self.k + sbert_ranks[i]))

            final_results.append({
                "doc_id": i,
                "rrf_score": rrf_score,
                "bm25_rank": bm25_ranks[i],
                "sbert_rank": sbert_ranks[i],
                "text": self.corpus[i]
            })

        # Sort by RRF score descending and return top_k
        final_results = sorted(final_results, key=lambda x: x["rrf_score"], reverse=True)
        return final_results[:top_k]

# Initialize the retriever instance
hybrid_retriever = HybridRetriever(corpus)
print("HybridRetriever class and instance created successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HybridRetriever class and instance created successfully.


In [14]:
# Cell 4: Cross-Encoder Re-Ranker (Part 3)
# Load the Cross-Encoder model (slower but more precise than Bi-Encoders)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    Takes the initial candidates and re-scores them using a Cross-Encoder.
    Input: Original query and the list of dicts from HybridRetriever.
    Returns: Top-k re-ranked documents with their new CE scores.
    """
    # 1. Prepare pairs for the Cross-Encoder: [ [query, doc1], [query, doc2], ... ]
    # We use the 'text' field from our candidate dictionaries.
    query_doc_pairs = [[query, cand["text"]] for cand in candidates]

    # 2. Predict relevance scores (logits)
    # Higher scores mean the document is more relevant to the query.
    ce_scores = cross_encoder.predict(query_doc_pairs)

    # 3. Attach scores to the candidate dictionaries
    for i, score in enumerate(ce_scores):
        candidates[i]["ce_score"] = float(score)

    # 4. Sort candidates by the new Cross-Encoder score in descending order
    reranked_docs = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)

    return reranked_docs[:top_k]

print(" Cross-Encoder model loaded and rerank function defined.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Cross-Encoder model loaded and rerank function defined.


In [20]:
# Cell 5: Mock HyDE (Bypassing API Quota)
def expand_query_hyde(query: str) -> str:
    """
    Returns a pre-defined hypothetical answer for specific test queries
    to avoid 429 RESOURCE_EXHAUSTED errors.
    """
    mock_responses = {
        "how do transformers encode meaning?": "Transformers use self-attention mechanisms to weigh the importance of different words in a sequence. This allows the model to capture context and encode semantic meaning through parallel processing of tokens.",
        "optimization techniques for training": "Optimization in neural networks involves using algorithms like Gradient Descent and Stochastic Gradient Descent (SGD) to minimize the loss function. These techniques update weights based on calculated gradients to improve model accuracy.",
        "what is the GELU activation function?": "The Gaussian Error Linear Unit (GELU) is a high-performance activation function used in transformer models like GPT and BERT. It weights inputs by their percentile, providing a smoother gradient than ReLU."
    }

    return mock_responses.get(query.lower(), query)

print("Mock HyDE Expansion defined for local testing.")

Mock HyDE Expansion defined for local testing.


In [16]:
# Cell 6: End-to-End Advanced RAG Pipeline (Part 5)
def advanced_rag(user_query: str) -> str:
    """
    Executes the full Advanced RAG pipeline:
    1. Query Expansion (HyDE)
    2. Hybrid Retrieval (BM25 + SBERT + RRF)
    3. Cross-Encoder Re-Ranking
    4. Grounded LLM Generation
    """

    # --- Step 1: Query Expansion (HyDE) ---
    # We use the expanded query for retrieval to better match technical documents.
    expanded_query = expand_query_hyde(user_query)

    # --- Step 2: Hybrid Retrieval ---
    # Retrieve top 10 candidates from the corpus using the expanded query.
    initial_candidates = hybrid_retriever.retrieve(expanded_query, top_k=10)

    # --- Step 3: Cross-Encoder Re-Ranking ---
    # Re-rank the candidates using the original user query for maximum precision.
    # We narrow down from 10 candidates to the top 3.
    final_docs = rerank(user_query, initial_candidates, top_k=3)

    # --- Step 4: Final Answer Generation ---
    # Format the top 3 documents into a context string for the LLM.
    context_text = "\n\n".join([f"[Document {i+1}]\n{d['text']}" for i, d in enumerate(final_docs)])

    gen_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a university academic assistant. Answer the question using ONLY the provided context. If the answer is not present, state that you do not have enough information. Keep the response concise."),
        ("human", "Context:\n{context}\n\nQuestion: {question}")
    ])

    generation_chain = gen_prompt | hyde_llm | StrOutputParser()

    answer = generation_chain.invoke({
        "context": context_text,
        "question": user_query
    })

    return answer

print("Advanced RAG pipeline function defined.")

Advanced RAG pipeline function defined.


In [21]:
# Cell 7: Final Comparison Experiment
test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is the GELU activation function?"
]

print(f"{'Query':<40} | {'Naïve RAG Top Doc':<40} | {'Advanced RAG Top Doc'}")
print("-" * 120)

for q in test_queries:
    # Baseline: SBERT only (Local)
    naive_doc = naive_rag_retrieve(q)

    # Advanced: Mock Expansion -> Hybrid Retrieval -> Re-Rank (Local)
    expanded = expand_query_hyde(q)
    candidates = hybrid_retriever.retrieve(expanded, top_k=10)
    reranked = rerank(q, candidates, top_k=1)
    advanced_doc = reranked[0]["text"]

    print(f"{q[:38]:<40} | {naive_doc[:38]:<40} | {advanced_doc[:38]}")

Query                                    | Naïve RAG Top Doc                        | Advanced RAG Top Doc
------------------------------------------------------------------------------------------------------------------------
how do transformers encode meaning?      | Transformers use self-attention mechan   | Transformers use self-attention mechan
optimization techniques for training     | Neural networks learn by adjusting wei   | Backpropagation calculates gradients t
what is the GELU activation function?    | The GELU activation function is often    | The GELU activation function is often 


## RAG Comparison Results

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
| :------------------------------------- | :------------------------------------- | :------------------------------------- | :---------------- |
| "how do transformers encode meaning?" | Transformers use self-attention... | Transformers use self-attention... | No |
| "optimization techniques for training" | Neural networks learn by adjusting... | Backpropagation calculates... | Yes |
| "what is the GELU activation function?" | The GELU activation function... | The GELU activation function... | No |